# LightGCN (Biased Evaluation) — Yahoo! R3

Baseline LightGCN on the **biased Yahoo! R3 logs** (`user.txt`) with a user-wise 80/20 train–test split.

- Dataset: `../data/yahoo_data/user.txt`
- Evaluation: RMSE/MAE on biased test split (Naive ERM)
- Output: `biased_results_yahoo_lightgcn.csv` in this folder.


In [1]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Lambda
from tensorflow.keras.models import Model

from sklearn.metrics import mean_squared_error, mean_absolute_error

print("TensorFlow version:", tf.__version__)


def save_biased_results_csv(
    csv_path,
    dataset_name,
    model_arch,
    estimator_name,
    y_true,
    y_pred,
    ndcg5=None,
    ndcg10=None,
    recall5=None,
    recall10=None,
):
    """Save biased evaluation metrics for one model into a CSV.

    Columns: Dataset, Architecture, Model, MSE, RMSE, MAE, NDCG@5, NDCG@10, Recall@5, Recall@10.
    """
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))

    row = {
        "Dataset": dataset_name,
        "Architecture": model_arch,
        "Model": estimator_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "NDCG@5": np.nan if ndcg5 is None else float(ndcg5),
        "NDCG@10": np.nan if ndcg10 is None else float(ndcg10),
        "Recall@5": np.nan if recall5 is None else float(recall5),
        "Recall@10": np.nan if recall10 is None else float(recall10),
    }

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    df.to_csv(csv_path, index=False)
    print(f"Saved biased results to: {csv_path}")


TensorFlow version: 2.20.0


In [2]:
# =========
# Load biased Yahoo logs and create user-wise 80/20 split
# =========

# From this notebook's location (biased data/yahoo), the Yahoo user.txt
# lives under ../../data/yahoo_data/user.txt inside pranathi3.
ratings = pd.read_csv('../../data/yahoo_data/user.txt', names=['userId', 'itemId', 'rating'])
print("Data loaded!", ratings.shape)

train_rows, test_rows = [], []

for user_id, user_data in ratings.groupby('userId'):
    user_data = user_data.sample(frac=1, random_state=42)
    n_items = len(user_data)
    train_size = max(1, int(0.8 * n_items))
    train_rows.append(user_data.iloc[:train_size])
    if train_size < n_items:
        test_rows.append(user_data.iloc[train_size:])

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows) if test_rows else train_df.sample(frac=0.2, random_state=42)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Encode user/item indices
user_ids = pd.concat([train_df['userId'], test_df['userId']]).unique()
item_ids = pd.concat([train_df['itemId'], test_df['itemId']]).unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {i: j for j, i in enumerate(item_ids)}

train_df['user'] = train_df['userId'].map(user2idx)
train_df['item'] = train_df['itemId'].map(item2idx)

test_df['user'] = test_df['userId'].map(user2idx)
test_df['item'] = test_df['itemId'].map(item2idx)

num_users = len(user2idx)
num_items = len(item2idx)

X_train = [train_df['user'].values, train_df['item'].values]
X_test = [test_df['user'].values, test_df['item'].values]

y_train = train_df['rating'].values.astype('float32')
y_test = test_df['rating'].values.astype('float32')

print("Encoded users:", num_users, "items:", num_items)


Data loaded! (311704, 3)
Train shape: (243340, 3)
Test shape: (68364, 3)
Encoded users: 15400 items: 1000


In [3]:
# =========
# LightGCN-style baseline model (embeddings + normalized dot product)
# =========

embedding_dim = 32

user_input = Input(shape=(1,), dtype='int32', name='user_input')
item_input = Input(shape=(1,), dtype='int32', name='item_input')

user_embedding = Embedding(num_users, embedding_dim, name='user_embedding')(user_input)
item_embedding = Embedding(num_items, embedding_dim, name='item_embedding')(item_input)

user_vec = Lambda(lambda x: tf.nn.l2_normalize(x, axis=-1))(user_embedding)
item_vec = Lambda(lambda x: tf.nn.l2_normalize(x, axis=-1))(item_embedding)

score = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=-1, keepdims=True))([user_vec, item_vec])

model = Model(inputs=[user_input, item_input], outputs=score)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │    492,800 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 32)     │     32,000 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 32)     │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 1, 32)     │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 1, 1)      │          0 │ lambda[0][0],     │
│                     │                   │            │ lambda_1[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 524,800 (2.00 MB)

 Trainable params: 524,800 (2.00 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# =========
# Train baseline LightGCN on biased Yahoo split
# =========

MODEL_PATH = "C:/Users/prana/OneDrive/Documents/thesis/pranathi3/biased data/yahoo/lightgcn_yahoo_best_model.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=4096,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - loss: 10.8462 - mae: 2.8824 - val_loss: 10.8268 - val_mae: 2.8766
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step - loss: 9.7133 - mae: 2.6773 - val_loss: 8.9951 - val_mae: 2.5368
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 110ms/step - loss: 7.6293 - mae: 2.2559 - val_loss: 7.0246 - val_mae: 2.1183
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 110ms/step - loss: 6.5180 - mae: 2.0007 - val_loss: 6.3916 - val_mae: 1.9666
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 117ms/step - loss: 6.1941 - mae: 1.9200 - val_loss: 6.2185 - val_mae: 1.9228
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - loss: 6.1043 - mae: 1.8968 - val_loss: 6.1710 - val_mae: 1.9105
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - loss: 6.0799 - mae: 1.8903 - val_loss: 6.1583 - val_mae: 1.9072
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 117ms/step - loss: 6.0734 - mae: 1.8886 - val_loss: 6.1549 - val_mae: 1.9064
Epoch 9/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 120ms/step - 

In [5]:
# =========
# Wrapper to compute and save biased metrics to CSV
# =========

def compute_and_save_biased_results_lightgcn_yahoo():
    """Compute Naive ERM metrics on biased Yahoo LightGCN split and save to CSV."""
    y_pred = model.predict(X_test, verbose=0).flatten()
    csv_path = "biased_results_yahoo_lightgcn.csv"
    return save_biased_results_csv(
        csv_path=csv_path,
        dataset_name="YAHOO",
        model_arch="LightGCN",
        estimator_name="Naive ERM",
        y_true=y_test,
        y_pred=y_pred,
    )

print("Call compute_and_save_biased_results_lightgcn_yahoo() after training to write CSV.")


Call compute_and_save_biased_results_lightgcn_yahoo() after training to write CSV.
